<a href="https://colab.research.google.com/github/danielaschuck/ACID-RESUME/blob/main/ACID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3

# conecta ao banco
conn = sqlite3.connect("banco.db")

# cria cursor
cursor = conn.cursor()

print("Conexão criada!")

Conexão criada!


Neste exemplo, **a Consistência** foi demonstrada por meio da restrição CHECK aplicada na coluna saldo da tabela contas. Essa regra garante que apenas valores maiores ou iguais a zero possam ser armazenados, impedindo a inserção de dados inválidos no banco de dados.

In [2]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS contas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    saldo REAL NOT NULL CHECK (saldo >= 0)
)
""")

conn.commit()

print("Tabela criada!")

Tabela criada!


In [4]:
# inserindo clientes

cursor.execute("""
INSERT INTO contas (nome, saldo)
VALUES ('Daniela', 1000)
""")

cursor.execute("""
INSERT INTO contas (nome, saldo)
VALUES ('Carlos', 500)
""")

conn.commit()

print("Clientes inseridos!")

Clientes inseridos!


In [5]:
cursor.execute("SELECT * FROM contas")

print(cursor.fetchall())

[(1, 'Daniela', 1000.0), (2, 'Carlos', 500.0)]


In [6]:
cursor.execute("""
INSERT INTO contas (nome, saldo)
VALUES ('Daniela', 1000)
""")

cursor.execute("""
INSERT INTO contas (nome, saldo)
VALUES ('Carlos', 500)
""")

conn.commit()

print("Usuários inseridos!")

Usuários inseridos!


In [7]:
cursor.execute("SELECT * FROM contas")

contas = cursor.fetchall()

for conta in contas:
    print(conta)

(1, 'Daniela', 1000.0)
(2, 'Carlos', 500.0)
(3, 'Daniela', 1000.0)
(4, 'Carlos', 500.0)


Neste exemplo,

 a **Atomicidade** foi demonstrada durante a tentativa de transferência bancária entre duas contas. Nós verificamos que a conta de Daniela não possuía saldo suficiente para realizar a transferência. Dessa forma, a transação foi interrompida e o comando ROLLBACK cancelou todas as alterações realizadas. **Isso garante que a operação aconteça completamente ou não aconteça, evitando inconsistências no banco de dados.**

In [9]:
try:
    conn.execute("BEGIN")

    # verifica saldo da Daniela
    cursor.execute("""
    SELECT saldo
    FROM contas
    WHERE id = 1
    """)

    saldo = cursor.fetchone()[0]#FIRST LINE IN FIRST POSITION(1000)

    valor_transferencia = 2000

    # verifica se possui saldo
    if saldo < valor_transferencia:
        raise Exception("Saldo insuficiente!")

    # remove dinheiro
    cursor.execute("""
    UPDATE contas
    SET saldo = saldo - ?
    WHERE id = 1
    """, (valor_transferencia,))

    # adiciona dinheiro
    cursor.execute("""
    UPDATE contas
    SET saldo = saldo + ?
    WHERE id = 2
    """, (valor_transferencia,))

    conn.commit()

    print("Transferência realizada!")

except Exception as erro:
    conn.rollback()

    print("Erro encontrado:")
    print(erro)

    print("Rollback executado!")

Erro encontrado:
Saldo insuficiente!
Rollback executado!


Neste exemplo,

 **o Isolamento** foi demonstrado por meio da simulação de duas transações acessando o mesmo saldo simultaneamente. Sem mecanismos de isolamento, ambas poderiam utilizar o mesmo valor ao mesmo tempo, causando inconsistências no banco de dados. O isolamento garante que transações concorrentes não interfiram umas nas outras.


In [ ]:
# Transação 1 lê o saldo
cursor.execute("""
SELECT saldo
FROM contas
WHERE id = 1
""")

saldo_transacao_1 = cursor.fetchone()[0]

print("Transação 1 leu saldo:", saldo_transacao_1)

# Transação 2 lê o mesmo saldo
cursor.execute("""
SELECT saldo
FROM contas
WHERE id = 1
""")

saldo_transacao_2 = cursor.fetchone()[0]

print("Transação 2 leu saldo:", saldo_transacao_2)

In [10]:
try:

    conn.execute("BEGIN")

    # leitura do saldo
    cursor.execute("""
    SELECT saldo
    FROM contas
    WHERE id = 1
    """)

    saldo = cursor.fetchone()[0]

    valor = 800

    # verifica se ainda existe saldo disponível
    if saldo >= valor:

        cursor.execute("""
        UPDATE contas
        SET saldo = saldo - ?
        WHERE id = 1
        """, (valor,))

        conn.commit()

        print("Transferência realizada com segurança!")

    else:
        print("Saldo insuficiente!")

except:
    conn.rollback()

Transferência realizada com segurança!


Neste exemplo, **a Durabilidade** foi demonstrada após a execução do comando COMMIT. Depois da confirmação da transação, a alteração do saldo foi salva permanentemente no banco de dados, garantindo que os dados continuem armazenados mesmo após o encerramento da aplicação.


In [12]:
try:

    conn.execute("BEGIN")

    cursor.execute("""
    UPDATE contas
    SET saldo = saldo + 300
    WHERE id = 2
    """)

    conn.commit()

    print("Alteração salva com sucesso!")

except:

    conn.rollback()
    print("Erro na transação!")

# mostra os dados após o commit
cursor.execute("SELECT * FROM contas")

print(cursor.fetchall())

Alteração salva com sucesso!
[(1, 'Daniela', 200.0), (2, 'Carlos', 1100.0), (3, 'Daniela', 1000.0), (4, 'Carlos', 500.0)]


In [ ]:
cursor.execute("SELECT * FROM contas")

contas = cursor.fetchall()

for conta in contas:
    print(conta)